# CytoTRACE Differentiation Potential Across Epithelial and Malignant States

Transfers the Hotspot-derived malignant states onto the full epithelial/tumour
object, computes CytoTRACE scores (CellRank `CytoTRACEKernel`) as a proxy for
differentiation potential, and compares scores across cell states, RECIST
treatment stages and patients.

In [ ]:
# !pip install --quiet scanpy cellrank scvelo leidenalg igraph statsmodels
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/tumour/pulmonary_epi_Lin/

In [ ]:
import os

import cellrank as cr
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import seaborn as sns
from cellrank.kernels import CytoTRACEKernel
from scipy.stats import kruskal, mannwhitneyu, pearsonr
from statsmodels.stats.multitest import multipletests

## Config

In [ ]:
full_input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06.h5ad"
states_input_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states.h5ad"

joined_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join.h5ad"
output_path = "sub_Epi_Tumor_Alveolar_raw_scvi_combine_TN_FDG_res06_mod_new_states_join_cytotrace.h5ad"


figures_output_dir = "./figures"
tables_output_dir = "./tables"

# label transfer: base cell-type labels overwritten by malignant states from 3a
base_label_col = "cell_type_new"
subset_state_col = "dominant_state_p75"
state_col = "final_state"
unassigned_label = "unassigned"

# preprocessing / CytoTRACE
counts_layer = "raw_counts"
target_sum = 1e4
moments_n_pcs = 30
moments_n_neighbors = 30
embedding_basis = "X_draw_graph_fa"  # FDG layout computed upstream
score_col = "ct_score"               # high = less differentiated
pseudotime_col = "ct_pseudotime"
# pseudo-velocity layers (X copies / dense smoothings); dropped before saving to cut file size
layers_to_drop = ["spliced", "unspliced", "Ms", "Mu"]

normal_state_order = [
    "Alveolar cell type I", "Alveolar cell type II", "Transitional",
    "Club cell", "Non-malignant epithelial", "Goblet cell",
]
malignant_state_order = [
    "1_Epithelial baseline", "2_Differentiated epithelial", "3_Devp_stem-like",
    "4_Inflammatory EMT", "6_Proliferative", "5_Neurod_EMT-like",
]
state_order = normal_state_order + malignant_state_order
state_colors = {
    "Malignant": "#023fa5",
    "Non-malignant epithelial": "#84bcf0",
    "Transitional": "#63ffac",
    "Alveolar cell type I": "#7d87b9",
    "Alveolar cell type II": "#bec1d4",
    "Club cell": "#4a6fe3",
    "Goblet cell": "#8e063b",
    "1_Epithelial baseline": "#8dd3c7",
    "2_Differentiated epithelial": "#bebada",
    "3_Devp_stem-like": "#80b1d3",
    "4_Inflammatory EMT": "#fccde5",
    "5_Neurod_EMT-like": "#bc80bd",
    "6_Proliferative": "#ffed6f",
    "unassigned": "grey",
}

recist_order = ["0_adj_normal", "1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]
treatment_order = ["1_treatment_naive", "3_on_treatment", "4_TKI_resistance"]

# patient display order for per-patient plots (adjacent-normal samples first)
donor_order = [
    "TN008_AN", "TN004_AN", "TN006_AN", "TN007_AN", "TN002_AN", "TN001_AN", "TN005_AN", "TN003_AN", "TN009_AN",
    "TN008", "TN004", "TN006", "TN007", "TN002", "TN001", "TN005", "TN003", "TN009",
    "P003", "P002", "P008", "P006", "P012", "P005", "P013", "P001", "P009", "P011",
    "R006", "R003", "R001", "R010", "R009", "R007", "R005",
]

# treatment-stage comparison within each malignant state
stats_unit = "patient"          # "patient" (median per patient) or "cell"
min_cells_per_patient = 20      # patients below this are dropped for that state
correction_method = "fdr_bh"    # applied to post-hoc tests within each state
alpha = 0.05

# marker panels for exploratory dotplots
marker_panels = {
    "diff": {"MUC21", "PGC", "SLC34A2", "CLDN18", "GDF15", "ELF3"},
    "stem": {"LGR5", "PBX1", "SOX5", "SOX6", "ETV5", "PAX7", "LMX1A", "ROR1", "ROR2",
             "FGF20", "SCN2A", "GNGT1", "RIMBP2"},
    "lin": {"NAPSA", "NKX2-1"},
}

os.makedirs(figures_output_dir, exist_ok=True)
os.makedirs(tables_output_dir, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.set_figure_params(frameon=False, dpi=100, dpi_save=150)
cr.settings.verbosity = 2

## 1. Transfer malignant state labels (Figure 2)

In [ ]:
adata = sc.read(full_input_path)
adata_states = sc.read(states_input_path)
adata.obs_names_make_unique()
adata_states.obs_names_make_unique()

shared = adata_states.obs_names.intersection(adata.obs_names)
assert len(shared) == adata_states.n_obs, "some state-annotated cells are missing from the full object"

adata.obs[state_col] = adata.obs[base_label_col].astype(str)
adata.obs.loc[shared, state_col] = adata_states.obs.loc[shared, subset_state_col].astype(str).values
adata.obs[state_col] = adata.obs[state_col].replace(["None", "nan"], unassigned_label)
del adata_states

missing_colors = set(adata.obs[state_col].unique()) - set(state_colors)
assert not missing_colors, f"states without a colour: {missing_colors}"
adata.obs[state_col] = adata.obs[state_col].astype("category")
adata.uns[f"{state_col}_colors"] = [state_colors[c] for c in adata.obs[state_col].cat.categories]

adata.obs[state_col].value_counts()

In [ ]:
sc.pl.embedding(adata, basis=embedding_basis, color=state_col, size=10, show=False)
plt.savefig(f"{figures_output_dir}/08_transfer_label_true.pdf", bbox_inches="tight")
plt.show()

adata.obs["highlight_unassigned"] = np.where(adata.obs[state_col] == unassigned_label, unassigned_label, "other")
sc.pl.embedding(adata, basis=embedding_basis, color="highlight_unassigned", size=5,
                palette={unassigned_label: "red", "other": "lightgrey"})

adata.write(joined_path)

## 2. Preprocess

`X` is reset to raw counts first so normalisation is never applied on top of
already log-normalised data. There are no spliced/unspliced counts, so `X` stands
in for both layers; `scv.pp.moments` only provides the neighbourhood smoothing
that `CytoTRACEKernel` needs.

In [ ]:
# adata = sc.read(joined_path)  # resume point
adata.X = adata.layers[counts_layer].copy()
sc.pp.normalize_total(adata, target_sum=target_sum)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata)
print(f"{adata.var['highly_variable'].sum()} highly variable genes")

def add_moments(ad):
    ad.layers["spliced"] = ad.X
    ad.layers["unspliced"] = ad.X
    scv.pp.moments(ad, n_pcs=moments_n_pcs, n_neighbors=moments_n_neighbors)
    return ad

adata = add_moments(adata)
adata_hvg = add_moments(adata[:, adata.var["highly_variable"]].copy())

## 3. Compute CytoTRACE scores

All genes are used for the primary score (CytoTRACE relies on global transcriptional
diversity, Gulati et al., Science 2020). An HVG-only run is kept as a sensitivity check.

In [ ]:
ctk_all = CytoTRACEKernel(adata).compute_cytotrace()          # writes ct_score / ct_pseudotime to adata.obs
ctk_hvg = CytoTRACEKernel(adata_hvg).compute_cytotrace()
adata.obs["ct_score_hvg"] = adata_hvg.obs[score_col].reindex(adata.obs_names)
del adata_hvg

### Sanity check: all genes vs HVGs

In [ ]:
r, _ = pearsonr(adata.obs[score_col], adata.obs["ct_score_hvg"])
fig, ax = plt.subplots(figsize=(5, 5))
sns.regplot(x=adata.obs[score_col], y=adata.obs["ct_score_hvg"], ax=ax,
            scatter_kws={"s": 3, "alpha": 0.3, "rasterized": True}, line_kws={"color": "red"})
ax.set(xlabel="CytoTRACE (all genes)", ylabel="CytoTRACE (HVGs)", title=f"r = {r:.3f}")
sns.despine()
plt.savefig(f"{figures_output_dir}/10_cytotrace_all_vs_hvg.pdf", bbox_inches="tight")
plt.show()

## 4. CytoTRACE on the FDG layout

Per-stage panels share one colour scale so they are comparable across stages.

In [ ]:
sc.pl.embedding(adata, basis=embedding_basis, color=[pseudotime_col, score_col],
                color_map="Spectral_r", size=5, show=False)
plt.savefig(f"{figures_output_dir}/09_cytotrace_score_pseudotime.pdf", bbox_inches="tight")
plt.show()

vmin = [adata.obs[pseudotime_col].min(), adata.obs[score_col].min()]
vmax = [adata.obs[pseudotime_col].max(), adata.obs[score_col].max()]
for stage in recist_order:
    sub = adata[adata.obs["recist_v3"] == stage]
    if sub.n_obs == 0:
        continue
    sc.pl.embedding(sub, basis=embedding_basis, color=[pseudotime_col, score_col],
                    color_map="Spectral_r", size=5, vmin=vmin, vmax=vmax,
                    title=[f"{pseudotime_col} | {stage}", f"{score_col} | {stage}"], show=False)
    plt.savefig(f"{figures_output_dir}/18_cytotrace_{stage}.pdf", bbox_inches="tight")
    plt.show()

## 5. CytoTRACE by cell state and treatment stage


In [ ]:
def plot_state_violin(obs_df, y, states, stages=None, colors=state_colors, save_path=None):
    """Violin of `y` per state, optionally split by RECIST stage; coloured by state."""
    df = obs_df[[state_col, "recist_v3", y]].copy()
    df[state_col], df["recist_v3"] = df[state_col].astype(str), df["recist_v3"].astype(str)
    df = df[df[state_col].isin(states)]

    if stages is None:
        df["group"] = df[state_col]
        order = [s for s in states if s in set(df["group"])]
        palette = {g: colors[g] for g in order}
    else:
        df = df[df["recist_v3"].isin(stages)]
        df["group"] = df[state_col] + " | " + df["recist_v3"]
        present = set(df["group"])
        order = [f"{s} | {t}" for s in states for t in stages if f"{s} | {t}" in present]
        palette = {g: colors[g.split(" | ")[0]] for g in order}

    fig, ax = plt.subplots(figsize=(max(4.5, 0.35 * len(order)), 5))
    sns.violinplot(data=df, x="group", y=y, hue="group", order=order, hue_order=order,
                   palette=palette, legend=False, density_norm="width", inner="box",
                   cut=0, linewidth=0.5, ax=ax)
    ax.set(xlabel="", ylabel=y)
    ax.tick_params(axis="x", rotation=90)
    sns.despine()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

In [ ]:
plot_state_violin(adata.obs, pseudotime_col, state_order,
                  save_path=f"{figures_output_dir}/11_ct_pseudotime_by_state.pdf")
plot_state_violin(adata.obs, score_col, state_order,
                  save_path=f"{figures_output_dir}/12_ct_score_by_state.pdf")
plot_state_violin(adata.obs, pseudotime_col, state_order, stages=recist_order,
                  save_path=f"{figures_output_dir}/13_ct_pseudotime_by_state_stage.pdf")
plot_state_violin(adata.obs, score_col, normal_state_order, stages=recist_order,
                  save_path=f"{figures_output_dir}/14_ct_score_by_normal_state_stage.pdf")
plot_state_violin(adata.obs, score_col, malignant_state_order, stages=treatment_order,
                  save_path=f"{figures_output_dir}/15_ct_score_by_malignant_state_stage.pdf")

## 6. Treatment-stage comparison within malignant states/ stats test 

In [ ]:
df = adata.obs[[state_col, "recist_v3", "ID_2", score_col]].copy()
for c in (state_col, "recist_v3", "ID_2"):
    df[c] = df[c].astype(str)
df = df[df[state_col].isin(malignant_state_order) & df["recist_v3"].isin(treatment_order)]

if stats_unit == "patient":
    grouped = df.groupby([state_col, "recist_v3", "ID_2"])[score_col]
    df = pd.concat([grouped.median(), grouped.size().rename("n_cells")], axis=1).reset_index()
    df = df[df["n_cells"] >= min_cells_per_patient]

rows = []
for state in malignant_state_order:
    d = df[df[state_col] == state]
    groups = {t: d.loc[d["recist_v3"] == t, score_col].values for t in treatment_order}
    groups = {t: v for t, v in groups.items() if len(v) >= 2}
    n = {t: len(v) for t, v in groups.items()}
    if len(groups) < 2:
        rows.append({"state": state, "test": "skipped", "comparison": "<2 groups with n>=2", "n": str(n)})
        continue

    h, p_omni = kruskal(*groups.values())
    rows.append({"state": state, "test": "kruskal", "comparison": "all", "statistic": h, "p": p_omni, "n": str(n)})
    if p_omni >= alpha:
        continue

    stages = list(groups)
    posthoc = []
    for i in range(len(stages)):
        for j in range(i + 1, len(stages)):
            a, b = stages[i], stages[j]
            u, p = mannwhitneyu(groups[a], groups[b], alternative="two-sided")
            posthoc.append({"state": state, "test": "mannwhitneyu", "comparison": f"{a} vs {b}",
                            "statistic": u, "p": p, "n": f"{n[a]} vs {n[b]}",
                            "median_a": np.median(groups[a]), "median_b": np.median(groups[b])})
    p_adj = multipletests([r["p"] for r in posthoc], method=correction_method)[1]
    for r, pa in zip(posthoc, p_adj):
        r["p_adj"] = pa
    rows.extend(posthoc)

stats_df = pd.DataFrame(rows)
stats_df.insert(0, "unit", stats_unit)
stats_df.to_csv(f"{tables_output_dir}/cytotrace_treatment_stage_stats_{stats_unit}.csv", index=False)
stats_df

## 7. Save CytoTRACE output

`X` is restored to raw counts and the pseudo-velocity layers are dropped; these are
copies or dense smoothings of `X` and account for most of the original ~66 GB file.
Recompute moments (section 2) if the kernel is needed again.

In [ ]:
adata_out = adata.copy()
adata_out.X = adata_out.layers[counts_layer].copy()
for layer in layers_to_drop:
    adata_out.layers.pop(layer, None)
if "log1p" in adata_out.uns:
    del adata_out.uns["log1p"]

adata_out.write(output_path, compression="gzip")
print(f"{output_path}: {os.path.getsize(output_path) / 1e9:.1f} GB")
del adata_out